# 02 · Scoring — what "correct" actually means

**The thesis: an accuracy number is only as trustworthy as the string
comparison behind it.** This notebook shows the scorer, the real edge cases it
has to survive, and three genuine bugs we found in it — each of which silently
changed the headline number.

No API key needed.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `is_correct(p,g,q)` | score a prediction against the gold answer | `is_correct('48 nucleotides','48')` → True |
| `_num(s)` | pull the numeric value out of a string | `'3.42E-04'` → 0.000342 |
| `_close(a,b)` | numeric match with relative tolerance | `14.47` ≈ `14.5` |

## Step 1 — the simple case

Models answer in prose. The gold is terse. A naive string equality would score
a correct answer as wrong, so numbers are compared **as numbers**.

In [1]:
import sys; sys.path.insert(0, "")
from scorer import is_correct, _num, _close

print(is_correct("48 nucleotides", "48"), " <- same number, different words")
print(is_correct("The answer is 27", "27 donors"), " <- number extracted from a sentence")
print(is_correct("46", "48"), " <- genuinely different, stays False")

True  <- same number, different words
True  <- number extracted from a sentence
False  <- genuinely different, stays False


## Step 2 — bug one: scientific notation

Our first `_num` used the pattern `-?\d+(\.\d+)?`, which reads `3.42E-04` as
**3.42** — the exponent is silently dropped. The model was right and we scored
it wrong. Compare the parse below against what the string actually means.

In [2]:
print("_num('3.42E-04') =", _num("3.42E-04"), " (0.000342, not 3.42)")
print("_num('14.5%')    =", _num("14.5%"))
print("_num('-3 bp')    =", _num("-3 bp"))

_num('3.42E-04') = 0.000342  (0.000342, not 3.42)
_num('14.5%')    = 14.5
_num('-3 bp')    = -3.0


## Step 3 — bug two: rounding

Gold answers are written to the precision the source paper used. A computed
value will not match to 6 decimal places. An absolute tolerance marks a correct
answer wrong, so we compare with a **relative** tolerance of 1%.

In [3]:
print(is_correct("3.42E-04", "0.00034", "x"), " <- rounding, should be True")
print(is_correct("14.47%",   "14.5%",   "x"), " <- rounding, should be True")
print(is_correct("17",       "51",      "x"), " <- not close, stays False")

True  <- rounding, should be True
True  <- rounding, should be True
False  <- not close, stays False


## Step 4 — bug three: sign on "difference" questions

The benchmark writes differences **unsigned**: for lengths 20 and 23 the gold
is `3`, not `-3`. A model answering `-3` has done the task correctly. We allow
magnitude matching *only* when the question asks for a difference — everywhere
else the sign still has to be right.

In [4]:
q = "What is the difference in length between the forward and reverse primers?"
print(is_correct("-3 bp", "3 nucleotides", q),         " <- difference question: True")
print(is_correct("-3",    "3",   "What is the total?"), " <- NOT a difference: stays False")

True  <- difference question: True
False  <- NOT a difference: stays False


## Step 5 — non-numeric answers

Some golds are text. Those fall through to a normalised string comparison:
lowercase, strip punctuation and spacing, then compare.

In [5]:
print(is_correct("TC HDL - (TG 5)", "TC HDL - (TG/5)"))
print(is_correct("yes, arthritis", "Yes, arthritis"))
print(is_correct("completely different text", "yes, arthritis"))

True
True
False


## Step 6 — the honest limitation

This scorer is deliberately strict-but-fair, and it is **not** a judge of free
text. A long prose answer that happens to contain the right number will pass; a
right answer phrased very differently may fail. Any number we publish states
which scorer produced it, and an LLM judge can be layered on later for the
free-text cases.

State this in the paper rather than hoping a reviewer does not ask.

In [6]:
# a case the scorer gets WRONG - worth knowing before you trust a number
print(is_correct("The value is somewhere around 27 or maybe 28", "27"),
      " <- passes, though the answer is hedged")

True  <- passes, though the answer is hedged


## What to carry out of this notebook

```
 · numbers are compared as numbers, with 1% relative tolerance
 · "difference" questions ignore sign; nothing else does
 · three real bugs here each moved the headline accuracy
 · the scorer is strict-but-fair, not an oracle - say so in the paper
```

Next: `03_baseline.ipynb` — the plain model call the agent has to beat.